# 07 · Deep learning: a small CNN on log-mel spectrograms

**Question:** does a CNN that learns its own features from the spectrogram beat the hand-made
features of notebooks 03-05 under exactly the same protocol (same 25 grouped splits, same metrics)?

**Input:** 64 log-mel bands over 0-4 kHz, 25 ms frames every 10 ms, a fixed 7 s per clip (crop or pad).
**Model:** 4 convolution blocks, pooling over time only (so pitch stays visible), dropout, one linear
layer; about 100k weights (`src/cnn.py`).
**Training,** inside each outer training part: AdamW, early stopping on a grouped validation part
carved out of the training part (the outer test fold is never used to pick the epoch), augmentation on
training batches only (time shift, low-level noise, SpecAugment). Two imbalance treatments:
square-root class-weighted loss, and a class-balanced sampler. A pitch-shift ablation (a shift along the
mel axis) runs on one repeat of the better treatment, because pitch may itself be a cue.

### How to run it on Google Colab
1. Here, once: `python scripts/make_colab_bundle.py` writes `data/colab/babycry_8k.zip` (the 479 clean
   clips, 8 kHz mono). Upload it to **My Drive/babycry/** in Google Drive.
2. Open this notebook in Colab (File → Open notebook → GitHub), choose **Runtime → Change runtime type → T4 GPU**,
   then **Runtime → Run all**. Allow Drive access when asked.
3. If the repository is private, add a read-only GitHub token as the Colab secret `GITHUB_TOKEN`
   (key icon in the left bar). It is read from the Secrets panel and never printed or saved in the notebook.
4. Results are saved to Drive after every split. If the session disconnects, run all cells again:
   finished splits are skipped.
5. Download from **My Drive/babycry/results/**: `predictions/cnn_*.csv` and `cnn_training_log.csv`, plus
   the executed notebook (File → Download → .ipynb). Put them in the repository's `results/` and
   `notebooks/` folders.

The notebook also runs locally (CPU, about 1-2 minutes per split): it then reads the audio through
`.env` like the other notebooks and keeps its per-split files in `data/cnn_runs/` (gitignored).

## 1. Setup

In [11]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/techatanu/Infant-State-Recognition-System.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/babycry")
    REPO = Path("/content/Infant-State-Recognition-System")
    if not REPO.exists():
        url = REPO_URL
        try:                                    # only needed for a private repository
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        done = subprocess.run(["git", "clone", "--quiet", url, str(REPO)], capture_output=True)
        if done.returncode != 0:                # never echo the URL: it may hold the token
            raise RuntimeError("git clone failed; for a private repo set the GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-dl.txt")], check=True)

    DATA = Path("/content/data")
    if not DATA.exists():                       # copy to VM's local disk so training I/O isn't bottlenecked by Drive
        zip_path = DRIVE / "babycry_8k.zip"
        folder_path = DRIVE / "babycry_8k"
        if zip_path.exists():
            shutil.copy(zip_path, "/content/babycry_8k.zip")
            shutil.unpack_archive("/content/babycry_8k.zip", DATA)
        elif folder_path.exists():
            shutil.copytree(folder_path, DATA / "babycry_8k")
        else:
            raise FileNotFoundError(f"Neither {zip_path} nor {folder_path} found in Google Drive.")

    # Point to the extracted folder (adjust to str(DATA / "babycry_8k") if your loader expects the inner folder directly)
    os.environ["BABYCRY_DATA_DIR"] = str(DATA)
    RUNS = DRIVE / "results" / "cnn_runs"
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUNS = REPO / "data" / "cnn_runs"
RUNS.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"python {sys.version.split()[0]}, torch {torch.__version__}, device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else f" ({torch.get_num_threads()} CPU threads)"))
print(f"repo commit {commit}, per-split results in {RUNS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
python 3.13.16, torch 2.11.0+cu130, device cuda (Tesla T4)
repo commit 3fd3a12, per-split results in /content/drive/MyDrive/babycry/results/cnn_runs


The CNN code is tested here with the same tests as on the development machine (they check, for
example, that the torch spectrogram equals the numpy reference and that the network learns pure tones).

In [12]:
import importlib.util

if importlib.util.find_spec("pytest") is None:
    print("pytest is not installed in this environment; skipping the CNN tests")
else:
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_cnn.py"], capture_output=True, text=True)
    print(tests.stdout.strip().splitlines()[-1] if tests.returncode == 0 else tests.stdout[-3000:])
    assert tests.returncode == 0, "CNN tests failed: see above"

9 passed in 37.31s


In [13]:
import warnings

from src.audio import CLASSES, load_audio, load_clean_index
from src.cnn import N_SAMPLES, VARIANTS, fix_length, prediction_table, train_split
from src.metrics import (balanced_accuracy, cluster_bootstrap, encode, interval, macro_f1, paired_bootstrap,
                         per_class_table, predictions_by_repeat, repeat_scores)
from src.splits import PREDICTIONS_DIR, load_folds, load_predictions, outer_splits, repeats, save_predictions

warnings.filterwarnings("ignore", message="The least populated class")   # 2-clip classes in the validation split
pd.set_option("display.width", 160)

def resolve_audio_path(p) -> Path:
    """Resolve audio file path across /content/data, /content/data/babycry_8k, or Google Drive."""
    p_obj = Path(p)
    if p_obj.exists():
        return p_obj

    parts = p_obj.parts
    for prefix in ("babycry_8k", "data"):
        if prefix in parts:
            rel = Path(*parts[parts.index(prefix) + 1:])
            break
    else:
        rel = p_obj

    search_roots = [
        Path(os.environ.get("BABYCRY_DATA_DIR", "/content/data")),
        Path("/content/data/babycry_8k"),
        Path("/content/data"),
    ]
    if IN_COLAB:
        search_roots.extend([DRIVE / "babycry_8k", DRIVE])

    for root in search_roots:
        for candidate in (root / p_obj, root / rel, root / rel.name):
            if candidate.exists():
                return candidate

    return p_obj

idx = load_clean_index()
folds = load_folds(idx)  # idx["path"] remains untouched so it matches metadata/folds.csv
t, groups = encode(idx["label"]), idx["source"].to_numpy()
start = time.time()
waves = np.stack([fix_length(load_audio(resolve_audio_path(p))) for p in idx["path"]])
print(f"{len(idx)} clips loaded as {waves.shape} float32 in {time.time() - start:.0f} s "
      f"({(idx['duration_s'] < 7).sum()} padded, {(idx['duration_s'] > 7).sum()} cropped to 7 s)")

479 clips loaded as (479, 56000) float32 in 0 s (272 padded, 25 cropped to 7 s)


## 2. Train on every split

Settings (from `src/cnn.py`, fixed before any run): batch 32, AdamW with learning rate 1e-3 and weight decay
1e-2, at most 80 epochs, stopping after 15 epochs without a better validation score. `REPEATS` can be cut to
`["r0"]` if the session budget is too small; the results then say so.

Each split's predictions and training curve are written to `RUNS` as soon as it finishes; splits already
there are skipped, so a disconnected session resumes where it stopped.

In [14]:
REPEATS = repeats(folds)                  # all 5; ["r0"] for a shorter run
PITCH_ABLATION = True                     # one repeat of the better treatment with a mel-axis shift of up to 2 bands


def run(variant, repeat_list, tag=None, **overrides):
    tag = tag or variant
    (RUNS / tag).mkdir(exist_ok=True)
    for r in repeat_list:
        for k, train, test in outer_splits(folds, r):
            out = RUNS / tag / f"{r}_fold{k}.csv"
            if out.exists():
                continue
            seed = int(r[1:]) * 10 + k
            proba, log, history = train_split(waves, t, groups, train, test, variant, seed, DEVICE, **overrides)
            history.to_csv(RUNS / tag / f"{r}_fold{k}_history.csv", index=False)
            pd.DataFrame([{**log, "tag": tag, "repeat": r, "fold": k, "commit": commit, "device": DEVICE}]).to_csv(
                RUNS / tag / f"{r}_fold{k}_log.csv", index=False)
            prediction_table(proba, t, test, r, k).to_csv(out, index=False, float_format="%.6g")
            print(f"{tag:28s} {r} fold {k}: best epoch {log['best_epoch']:2d} of {log['epochs_run']:2d}, "
                  f"{log['seconds']:4.0f} s")


def collect(tag):
    files = sorted((RUNS / tag).glob("r*_fold?.csv"))
    pred = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    log = pd.concat([pd.read_csv(f) for f in sorted((RUNS / tag).glob("r*_fold?_log.csv"))], ignore_index=True)
    return pred, log


for variant in VARIANTS:
    run(variant, REPEATS)

weighted_loss                r0 fold 0: best epoch  0 of 16,   10 s
weighted_loss                r0 fold 1: best epoch  0 of 16,    6 s
weighted_loss                r0 fold 2: best epoch  0 of 16,    6 s
weighted_loss                r0 fold 3: best epoch  5 of 21,    8 s
weighted_loss                r0 fold 4: best epoch  0 of 16,    6 s
weighted_loss                r1 fold 0: best epoch 26 of 42,   16 s
weighted_loss                r1 fold 1: best epoch  0 of 16,    6 s
weighted_loss                r1 fold 2: best epoch  1 of 17,    6 s
weighted_loss                r1 fold 3: best epoch  0 of 16,    6 s
weighted_loss                r1 fold 4: best epoch  0 of 16,    6 s
weighted_loss                r2 fold 0: best epoch  0 of 16,    6 s
weighted_loss                r2 fold 1: best epoch  0 of 16,    6 s
weighted_loss                r2 fold 2: best epoch  0 of 16,    6 s
weighted_loss                r2 fold 3: best epoch  0 of 16,    6 s
weighted_loss                r2 fold 4: best epo

In [15]:
predictions, logs = {}, []
for variant in VARIANTS:
    predictions[variant], log = collect(variant)
    logs.append(log)
    assert len(predictions[variant]) == len(idx) * len(REPEATS), f"{variant}: missing splits"
    save_predictions(f"cnn_{variant}", predictions[variant], idx)
training_log = pd.concat(logs, ignore_index=True)
training_log.to_csv(REPO / "results" / "cnn_training_log.csv", index=False)
if IN_COLAB:                               # copies to download from Drive
    (DRIVE / "results" / "predictions").mkdir(parents=True, exist_ok=True)
    for variant in VARIANTS:
        shutil.copy(PREDICTIONS_DIR / f"cnn_{variant}.csv", DRIVE / "results" / "predictions")
    shutil.copy(REPO / "results" / "cnn_training_log.csv", DRIVE / "results")
print(training_log.groupby("variant")[["best_epoch", "epochs_run", "seconds"]].describe().round(1).T)
print(f"commit(s) {sorted(training_log['commit'].unique())}, device(s) {sorted(training_log['device'].unique())}")

variant           balanced_sampler  weighted_loss
best_epoch count              25.0           25.0
           mean                2.7            2.2
           std                 7.8            6.3
           min                 0.0            0.0
           25%                 0.0            0.0
           50%                 0.0            0.0
           75%                 1.0            0.0
           max                35.0           26.0
epochs_run count              25.0           25.0
           mean               18.7           18.2
           std                 7.8            6.3
           min                16.0           16.0
           25%                16.0           16.0
           50%                16.0           16.0
           75%                17.0           16.0
           max                51.0           42.0
seconds    count              25.0           25.0
           mean                7.5            7.3
           std                 3.2            2.6


## 3. Scores, on the same footing as notebook 05

Means over repeats (± std), 95% cluster-bootstrap intervals over babies, and the paired difference from the
always-hungry baseline. GPU training is not bit-for-bit reproducible, so the spread over repeats is the
honest measure of run-to-run variation.

In [16]:
majority_pred = load_predictions("majority", idx).loc[lambda d: d["repeat"].isin(REPEATS)]
majority = predictions_by_repeat(majority_pred, len(idx))


def evaluate(pred, n_boot=1000):
    per_rep = repeat_scores(pred)
    preds = predictions_by_repeat(pred, len(idx))
    row = {m: per_rep[m].mean() for m in per_rep.columns}
    row.update({f"{m}_std": per_rep[m].std(ddof=1) for m in per_rep.columns})
    for m, fn in [("balanced_accuracy", balanced_accuracy), ("macro_f1", macro_f1)]:
        row[f"{m}_low"], row[f"{m}_high"] = interval(cluster_bootstrap(t, preds, groups, fn, n_boot, seed=0))
    d = paired_bootstrap(t, preds, majority, groups, n_boot=n_boot, seed=0)
    row["ba_vs_majority"], (row["ba_vs_majority_low"], row["ba_vs_majority_high"]) = d.mean(), interval(d)
    return row


rows = [{"model": "majority", "variant": "baseline", **evaluate(majority_pred)}]
rows += [{"model": "cnn", "variant": v, **evaluate(p)} for v, p in predictions.items()]
scores = pd.DataFrame(rows).set_index(["model", "variant"])
pd.DataFrame({
    "balanced acc": scores["balanced_accuracy"].map("{:.3f}".format) + " ± " + scores["balanced_accuracy_std"].map("{:.3f}".format),
    "BA 95% CI": scores["balanced_accuracy_low"].map("{:.3f}".format) + "-" + scores["balanced_accuracy_high"].map("{:.3f}".format),
    "macro-F1": scores["macro_f1"].map("{:.3f}".format),
    "BA vs majority (95% CI)": scores["ba_vs_majority"].map("{:+.3f}".format) + " (" + scores["ba_vs_majority_low"].map("{:+.3f}".format)
                               + ", " + scores["ba_vs_majority_high"].map("{:+.3f}".format) + ")",
    "accuracy": scores["accuracy"].round(3), "log-loss": scores["log_loss"].round(3), "top-3": scores["top3_accuracy"].round(3),
})

balanced acc    BA 95% CI macro-F1  BA vs majority (95% CI)  accuracy  log-loss  top-3
model    variant                                                                                                  
majority baseline          0.125 ± 0.000  0.125-0.143    0.111  +0.000 (+0.000, +0.000)     0.797     0.887  0.900
cnn      weighted_loss     0.150 ± 0.013  0.113-0.206    0.115  +0.026 (-0.012, +0.073)     0.491     1.715  0.831
         balanced_sampler  0.136 ± 0.030  0.099-0.177    0.054  +0.010 (-0.026, +0.047)     0.125     2.230  0.425

In [17]:
best_variant = scores.loc["cnn"]["balanced_accuracy"].idxmax()
pred = predictions[best_variant]
tables = [per_class_table(encode(g.sort_values("clip")["y_true"]), encode(g.sort_values("clip")["y_pred"]), groups)
          for _, g in pred.groupby("repeat")]
per_class = pd.concat([tbl[["predicted", "precision", "recall", "f1"]] for tbl in tables]).groupby(level=0, sort=False).mean()
per_class.insert(0, "babies", tables[0]["babies"])
per_class.insert(0, "clips", tables[0]["clips"])
per_class["reliable (5+ babies)"] = tables[0]["reliable"]
print(f"per class, {best_variant} (mean over repeats):")
per_class.round(3)

per class, weighted_loss (mean over repeats):


,clips,babies,predicted,precision,recall,f1,reliable (5+ babies)
belly_pain,16,7,30.8,0.012,0.038,0.016,True
burping,9,8,6.6,0.000,0.000,0.000,True
cold_hot,7,3,18.6,0.000,0.000,0.000,False
discomfort,21,14,13.8,0.028,0.048,0.025,True
hungry,382,205,282.2,0.799,0.591,0.675,True
lonely,11,8,63.8,0.082,0.400,0.129,True
scared,5,4,0.0,NaN,0.000,0.000,False
tired,28,17,63.2,0.058,0.121,0.075,True


## 4. Format check (as in notebook 06)

*cold_hot*, *lonely* and *scared* exist only as 44.1 kHz recordings. If the CNN uses the recording format, the
few 44.1 kHz clips of *tired*, *discomfort* and *burping* get more probability on those three classes than the
8 kHz clips of the same classes do. AUROC 0.5 = no format effect; the random forest in notebook 06 scored 0.81.

In [18]:
from sklearn.metrics import roc_auc_score

three = ["p_cold_hot", "p_lonely", "p_scared"]
for variant, p in predictions.items():
    p3 = p.assign(p3=p[three].sum(axis=1)).groupby("clip")["p3"].mean()
    check = idx.join(p3).loc[lambda d: d["label"].isin(["tired", "discomfort", "burping"])]
    is44 = check["sample_rate"] == 44100
    print(f"{variant:18s} mean p(cold_hot+lonely+scared): 44.1 kHz {check.loc[is44, 'p3'].mean():.3f}, "
          f"8 kHz {check.loc[~is44, 'p3'].mean():.3f}; AUROC {roc_auc_score(is44, check['p3']):.2f} "
          f"({is44.sum()} vs {(~is44).sum()} clips)")

weighted_loss      mean p(cold_hot+lonely+scared): 44.1 kHz 0.325, 8 kHz 0.273; AUROC 0.65 (6 vs 52 clips)
balanced_sampler   mean p(cold_hot+lonely+scared): 44.1 kHz 0.459, 8 kHz 0.373; AUROC 0.71 (6 vs 52 clips)


## 5. Pitch-shift ablation

The better treatment again on repeat r0 only, with a random shift of up to ±2 mel bands added to the
augmentation (roughly ±3-6% in frequency in the cry range). It is compared with the same 5 splits without the shift.

In [19]:
if PITCH_ABLATION:
    tag = f"{best_variant}_pitch"
    run(best_variant, ["r0"], tag=tag, pitch_shift_bands=2)
    pitch, pitch_log = collect(tag)
    pitch_log.to_csv(REPO / "results" / "cnn_pitch_ablation_log.csv", index=False)
    save_predictions(f"cnn_{tag}", pitch, idx)
    if IN_COLAB:
        shutil.copy(PREDICTIONS_DIR / f"cnn_{tag}.csv", DRIVE / "results" / "predictions")
        shutil.copy(REPO / "results" / "cnn_pitch_ablation_log.csv", DRIVE / "results")
    same = predictions[best_variant].loc[lambda d: d["repeat"] == "r0"]
    order = lambda d: d.sort_values("clip")
    a, b = order(pitch), order(same)
    d = paired_bootstrap(t, encode(a["y_pred"]), encode(b["y_pred"]), groups, n_boot=1000, seed=0)
    print(f"repeat r0: BA without shift {balanced_accuracy(t, encode(b['y_pred'])):.3f}, "
          f"with shift {balanced_accuracy(t, encode(a['y_pred'])):.3f}, "
          f"difference {d.mean():+.3f} (95% CI {interval(d)[0]:+.3f}, {interval(d)[1]:+.3f})")

weighted_loss_pitch          r0 fold 0: best epoch  0 of 16,    6 s
weighted_loss_pitch          r0 fold 1: best epoch  0 of 16,    6 s
weighted_loss_pitch          r0 fold 2: best epoch  0 of 16,    6 s
weighted_loss_pitch          r0 fold 3: best epoch 11 of 27,   11 s
weighted_loss_pitch          r0 fold 4: best epoch  0 of 16,    6 s
repeat r0: BA without shift 0.164, with shift 0.172, difference +0.019 (95% CI -0.044, +0.123)


## 6. Findings

Run on Google Colab (Tesla T4, Python 3.13.16, torch 2.11.0+cu130) at commit 3fd3a12, all 50 trainings
plus the 5 ablation splits. All 9 CNN tests passed there. On Colab, the setup cell and the audio-loading cell
were edited so that the audio can also come from an unzipped `babycry_8k` folder on Drive. Only data loading changed;
the model, training and scoring code is the committed `src/cnn.py`.

- **Early stopping kept the network after its first epoch.** The best epoch was epoch 0 in most splits
  (median 0 for both treatments; mean 2.2 and 2.7, at most 26 and 35). After one pass over the training
  clips the validation score only got worse. What the network learns about the training babies does not carry
  over to babies it has not heard.
- **No treatment beats the majority baseline.** Square-root weighted loss: balanced accuracy 0.150 ± 0.013
  (95% CI 0.113-0.206), +0.026 (-0.012, +0.073) vs majority. Class-balanced sampler: 0.136 ± 0.030, +0.010
  (-0.026, +0.047). Both intervals include 0. The weighted-loss CNN is nominally above the best classical model
  (random forest, 0.131), but the gap is inside the noise floor of notebook 04 (about 0.035-0.04).
- **The probabilities are worse than the class prior.** Log-loss is 1.715 and 2.230 against the baseline's 0.887.
  The balanced sampler rarely predicts hungry: accuracy 0.125, macro-F1 0.054.
- **Per class (weighted loss),** *lonely* has recall 0.400 but precision 0.082 (63.8 lonely predictions per repeat
  for 11 clips). *Hungry* recall drops to 0.591. Every other class has recall of at most 0.121, and *scared* is never predicted.
- **The format check is weaker than for the random forest but not zero:** AUROC 0.65 (weighted loss) and 0.71
  (sampler), against 0.81 in notebook 06. *Lonely* exists only as 44.1 kHz recordings, so its recall may partly
  reflect the recording format rather than the cry. Notebook 08 checks this.
- **Pitch-shift ablation:** on repeat r0, 0.164 without and 0.172 with a mel-axis shift, a paired difference of
  +0.019 (-0.044, +0.123). There is no evidence either way.